# 04 — DistilBERT Prosodic Boundary Classifier (Multi-corpus)

Fine-tunes `distilbert-base-uncased` on prosodic boundary labels from one or more corpora.

**Task:** Multi-task token classification — boundary (`b`), intonation type (`i`), break index (`x`).

**Baseline to beat:** PSST text-only GPT-Neo F1 = 0.77

**Fixed test set:** SBC001–005 for boundary detection and intonation at boundaries (gold conversational, held out from all training). The entire Boston University Radio News Corpus for break index verifcation and testing.

Disclaimer: Code was largely generated with the help of Claude Sonnet 4.6 (Anthropic, 2026). Prompts, code tweaks and verification by me.

---

## How to use this notebook

Run cells **top to bottom**. Cells you will edit regularly:

| Cell | Purpose |
|---|---|
| **Cell 1** | Paths, hyperparameter defaults, corpus load flags |
| **Cell 4** | Corpus loading (runs automatically based on flags) |
| **Cell 14** | Run queue — define which datasets + overrides per run |

Everything else runs unchanged between experiments.

---

## Corpus registry

Corpora are loaded in Cell 4 into a `CORPUS_REGISTRY` dict, then referenced by name in the run queue:

| Key | Corpus | Notes |
|---|---|---|
| `"libri"` | LibriTTS clean-100 + clean-360 | Silver-standard, read speech, ~145k samples |
| `"sbc"` | SBCSAE SBC006–SBC060 | Gold-standard, conversational (SBC001–005 are test-only) |
| `"bu"` | BU Radio News Corpus | Gold-standard, broadcast news, ~426 samples |

Add future corpora by adding a flag + load snippet in Cell 4 and a new key to `CORPUS_REGISTRY`.

---

## Output folder structure

```
models/
└── {run_notes}/                       # name of run
    ├── checkpoint/                    ← model weights + tokenizer
    ├── {run_id}_hparams.json
    ├── {run_id}_test_predictions.json
    ├── {run_id}_curves.png
    └── {run_id}_confusion_matrix.png
```


---
## Section 1 · Configuration

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — CONFIGURATION                                      ║
# ║  Edit paths, hyperparameter defaults, and corpus flags here. ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json, hashlib
from datetime import datetime

# ── Paths ─────────────────────────────────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/Capstone/project"
RUNS_ROOT  = f"{DRIVE_ROOT}/models"
LABELS_ROOT  = f"{DRIVE_ROOT}/labels"

# Corpus label roots — add new corpora here
BATCHED_ROOT_LIBRI_100 = f"{LABELS_ROOT}/clean-100"
BATCHED_ROOT_LIBRI_360 = f"{LABELS_ROOT}/clean-360"
BATCHED_ROOT_SBC       = f"{LABELS_ROOT}/sbcsae"
BATCHED_ROOT_BU        = f"{LABELS_ROOT}/bu"
BATCHED_ROOT_PS        = f"{LABELS_ROOT}/ps"
# BATCHED_ROOT_NEWCORPUS = f"{DRIVE_ROOT}/labels/newcorpus"

# ── Corpus load flags ─────────────────────────────────────────────────────────
# Set False to skip loading a corpus (saves time if you won't use it this session).
# A run that requests a skipped corpus will raise a KeyError.
LOAD_LIBRI = True   # ~145k samples — slow to load
LOAD_SBC   = True   # ~4k samples
LOAD_BU    = True   # ~426 samples
LOAD_PS    = True   # People's Speech silver-standard
# LOAD_NEWCORPUS = True

# SBC test holdout IDs — these are NEVER added to the registry or any training set
SBC_TEST_IDS = set(f"SBC{str(n).zfill(3)}" for n in range(1, 6))  # SBC001–SBC005

# ── Data split (train/val only — test is always SBC001–005) ───────────────────
SPLIT_SEED = 42
TRAIN_FRAC = 0.80
VAL_FRAC   = 0.10

# ── Text pre-processing ───────────────────────────────────────────────────────
STRIP_PUNCTUATION = False
EXTRA_FEATURES    = []

# ── POS feature flags (defaults — can be overridden per-run in Cell 14) ───────
# TRAIN_ON_TEXT : DistilBERT receives the real word sequence as input.
# TRAIN_ON_POS  : POS embeddings are injected post-transformer (combined mode).
#                 Requires TRAIN_ON_TEXT=True. POS-only mode (TRAIN_ON_TEXT=False,
#                 TRAIN_ON_POS=True) replaces input words with POS abbreviations.
# Neither True  : invalid — raises an error in run_experiment().
# Default here is text-only; override per run in the RUNS queue (Cell 14).
TRAIN_ON_TEXT     = True
TRAIN_ON_POS      = False
POS_EMBEDDING_DIM = 64      # embedding size before Linear projection to H=768
                             # only used in combined mode (text+POS)

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 3
WARMUP_STEPS  = 0
WEIGHT_DECAY  = 0.01

# ── Class-imbalance strategy (boundary head only) ─────────────────────────────
IMBALANCE_STRATEGY   = "none"
BOUNDARY_LOSS_WEIGHT = 5.0

# ── Multi-task loss weights ───────────────────────────────────────────────────
BOUNDARY_TASK_WEIGHT   = 1.0
INTONATION_TASK_WEIGHT = 1.0
BREAK_IDX_TASK_WEIGHT  = 1.0

# ── Run metadata ──────────────────────────────────────────────────────────────
RUN_NOTES = ""

# ── Dual checkpoint (optional) ────────────────────────────────────────────────
# When True, also saves the checkpoint with best val intonation F1 and runs a
# second test evaluation pass. Off by default — enable per-run via overrides.
SAVE_INTONATION_CHECKPOINT = True

# ── Model registry ─────────────────────────────────────────────────────────
# Tracks every run's deterministic name → parameters. Lives at RUNS_ROOT
# (parent of full/ and partial/), keyed by auto-incrementing model number.
MODEL_REGISTRY_PATH = f"{RUNS_ROOT}/model_registry.json"

print("✓ Configuration loaded.")
print(f"  Corpus flags:  LIBRI={LOAD_LIBRI}  SBC={LOAD_SBC}  BU={LOAD_BU}  PS={LOAD_PS}")
print(f"  POS flags:     TRAIN_ON_TEXT={TRAIN_ON_TEXT}  TRAIN_ON_POS={TRAIN_ON_POS}  POS_EMB_DIM={POS_EMBEDDING_DIM}")
print(f"  SBC test IDs:  {sorted(SBC_TEST_IDS)}")
print(f"  Dual checkpoint: {SAVE_INTONATION_CHECKPOINT}")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Deterministic run naming                                    ║
# ║  Computes a canonical name from dataset list + key params.   ║
# ║  Same inputs always produce the same name — no manual typing.║
# ╚══════════════════════════════════════════════════════════════╗

_DATASET_LETTERS = {"libri": "L", "ps": "P", "sbc": "S"}
_FULL_SET = {"libri", "ps", "sbc"}

_DEFAULT_LR      = 2e-5
_DEFAULT_WARMUP  = 0

def _format_lr(lr):
    """2e-05 → 'lr2e5'   1e-05 → 'lr1e5'   3e-05 → 'lr3e5'"""
    s = f"{lr:.0e}"            # e.g. "2e-05"
    mantissa, exp = s.split("e")
    exp_num = exp.lstrip("+-").lstrip("0") or "0"
    return f"lr{mantissa}e{exp_num}"

def compute_run_name(datasets: list[str], run_cfg: dict) -> tuple[str, str]:
    # ── Dataset letters, canonical order ───────────────────────────────────
    letters = "".join(
        _DATASET_LETTERS[d] for d in ["libri", "ps", "sbc"] if d in datasets
    )
    # ── Loss suffix (always present) ────────────────────────────────────────
    loss_suffix = "wl" if run_cfg["IMBALANCE_STRATEGY"].startswith("weight") else "stl"
    parts = [letters, loss_suffix]

    # ── POS suffix (optional) ───────────────────────────────────────────────
    train_text = run_cfg["TRAIN_ON_TEXT"]
    train_pos  = run_cfg["TRAIN_ON_POS"]
    if train_pos and not train_text:
        parts.append("posonly")
    elif train_pos and train_text:
        parts.append("pos")
    # else: text-only, default, no suffix

    # ── Punctuation suffix (optional) ───────────────────────────────────────
    if not run_cfg["STRIP_PUNCTUATION"]:
        parts.append("pp")
    # else: stripped, default, no suffix

    # ── Learning rate suffix (optional) ─────────────────────────────────────
    if run_cfg["LEARNING_RATE"] != _DEFAULT_LR:
        parts.append(_format_lr(run_cfg["LEARNING_RATE"]))

    # ── Warmup suffix (optional) ────────────────────────────────────────────
    if run_cfg["WARMUP_STEPS"] != _DEFAULT_WARMUP:
        parts.append(f"warmup{run_cfg['WARMUP_STEPS']}")

    name = "_".join(parts)
    category = "full" if set(datasets) == _FULL_SET else "partial"
    return name, category

print("✓ compute_run_name() defined.")

def resolve_run_path(name: str, category: str) -> tuple[str, str]:
    """
    Resolve the final run directory for a given deterministic name,
    appending a numeric disambiguator (_2, _3, ...) if the name is
    already taken on disk. Never overwrites an existing run directory.
    """
    base_dir = os.path.join(RUNS_ROOT, category)
    os.makedirs(base_dir, exist_ok=True)

    candidate = name
    suffix = 1
    while os.path.isdir(os.path.join(base_dir, candidate)):
        suffix += 1
        candidate = f"{name}_{suffix}"

    run_dir = os.path.join(base_dir, candidate)
    return candidate, run_dir

def _load_registry() -> dict:
    """Load model_registry.json, or return an empty dict if it doesn't exist yet."""
    if os.path.exists(MODEL_REGISTRY_PATH):
        with open(MODEL_REGISTRY_PATH) as f:
            return json.load(f)
    return {}


def _next_model_number(registry: dict) -> int:
    """Next available model number. Starts at 1 if registry is empty."""
    if not registry:
        return 1
    return max(int(k) for k in registry) + 1


def register_model(name: str, run_dir: str, datasets: list[str], run_cfg: dict,
                   loss_suffix_label: str) -> int:
    """
    Append a new entry to model_registry.json and return its assigned number.

    Parameters
    ----------
    name               : deterministic run name, e.g. "LPS_stl"
    run_dir             : full path to the run directory
    datasets            : list[str] used for training
    run_cfg             : resolved config dict (after overrides applied)
    loss_suffix_label   : "standard" or "weighted" — spelled out for the registry
                          (the folder/name suffix itself stays abbreviated as "stl"/"wl")
    """
    registry = _load_registry()
    model_num = _next_model_number(registry)

    rel_path = os.path.relpath(run_dir, RUNS_ROOT)

    registry[str(model_num)] = {
        "name": name,
        "path": rel_path,
        "datasets": datasets,
        "loss": loss_suffix_label,
        "train_on_text": run_cfg["TRAIN_ON_TEXT"],
        "train_on_pos": run_cfg["TRAIN_ON_POS"],
        "strip_punctuation": run_cfg["STRIP_PUNCTUATION"],
        "learning_rate": run_cfg["LEARNING_RATE"],
        "warmup_steps": run_cfg["WARMUP_STEPS"],
        "num_epochs": run_cfg["NUM_EPOCHS"],
        "run_notes": run_cfg["RUN_NOTES"],
    }

    with open(MODEL_REGISTRY_PATH, "w") as f:
        json.dump(registry, f, indent=2)

    return model_num

print("✓ resolve_run_path() defined.")
print("✓ Model registry functions defined.")

---
## Section 2 · Environment Setup

Run once per Colab session. Safe to re-run — all steps are idempotent.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Mount Drive + install packages                     ║
# ╚══════════════════════════════════════════════════════════════╝
from google.colab import drive
import os

drive.mount("/content/drive", force_remount=True)
os.makedirs(RUNS_ROOT, exist_ok=True)

!pip install -q transformers datasets scikit-learn matplotlib

import torch
print(f"\n✓ PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("⚠  No GPU — training will be slow.")
    print("   Runtime → Change runtime type → T4 GPU")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"   Device: {device}")


---
## Section 3 · Imports

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Imports                                            ║
# ╚══════════════════════════════════════════════════════════════╝
import os, json, string, random, shutil, traceback, gc, hashlib, subprocess
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from transformers import (
    DistilBertTokenizerFast,
    DistilBertModel,
    DistilBertPreTrainedModel,
    DistilBertConfig,
    get_linear_schedule_with_warmup,
)
from tqdm.notebook import tqdm, tqdm as tqdm_nb

class _NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.floating): return float(obj)
        if isinstance(obj, np.integer):  return int(obj)
        if isinstance(obj, np.ndarray):  return obj.tolist()
        return super().default(obj)

torch.manual_seed(SPLIT_SEED)
np.random.seed(SPLIT_SEED)
random.seed(SPLIT_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SPLIT_SEED)

print("✓ Imports complete.")


---
## Section 4 · Load Corpora

Loads each corpus based on the flags in Cell 1. Splits SBC into train (`sbc`) and fixed test holdout (`samples_sbc_test`) at load time.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Load corpora                                       ║
# ║                                                              ║
# ║  Loads each corpus based on flags set in Cell 1.             ║
# ║  Builds CORPUS_REGISTRY (training data only) and             ║
# ║  fixed evaluation sets (never in registry):                  ║
# ║    samples_sbc_test — boundary + intonation eval (SBC001–005)║
# ║    samples_bu       — break index eval (full BU corpus)      ║
# ║                                                              ║
# ║  To add a future corpus:                                     ║
# ║    1. Add LOAD_NEWCORPUS flag + BATCHED_ROOT_NEWCORPUS in C1 ║
# ║    2. Add a load block below (copy any existing block)       ║
# ║    3. Add key → samples to CORPUS_REGISTRY at the bottom     ║
# ╚══════════════════════════════════════════════════════════════╝

def load_label_files_batched(batched_root, label=""):
    """
    Load all samples from batch_*.json files in batched_root.

    Batch file layout (produced by annotation_pipeline_*.ipynb):
        batch_0000.json  ← dict keyed by sample_id, each value has "b", "i", "x"
        ...

    Returns
    -------
    samples : list[dict]  — keys: sample_id, tokens, boundary_labels,
                            intonation_labels, break_idx_labels
    meta    : dict        — empty placeholder (meta.json not bundled)
    """
    batch_files = sorted(
        f for f in os.listdir(batched_root)
        if f.startswith("batch_") and f.endswith(".json")
    )
    if not batch_files:
        raise FileNotFoundError(
            f"No batch files found in {batched_root}.\n"
            "Check path or run the relevant annotation pipeline first."
        )
    tag = f" [{label}]" if label else ""
    print(f"Found {len(batch_files)} batch files in {batched_root}{tag}.")

    samples, n_skipped = [], 0
    for fname in tqdm(batch_files, desc=f"Loading{tag}", unit="batch"):
        with open(os.path.join(batched_root, fname)) as f:
            batch = json.load(f)
            for sid, data in batch.items():
                b, i, x = data["b"], data["i"], data["x"]
                n = len(b["tokens"])
                x_labels = x["labels"] if x is not None else [""] * n
                if not (len(b["consensus"]) == len(i["labels"]) == len(x_labels) == n):
                    print(f"  ⚠ {sid}: mismatched label lengths — skipping.")
                    n_skipped += 1
                    continue
                samples.append({
                    "sample_id":         sid,
                    "tokens":            b["tokens"],
                    "boundary_labels":   b["consensus"],
                    "intonation_labels": i["labels"],
                    "break_idx_labels":  x_labels,
                })

    print(f"  ✓ {len(samples):,} samples loaded  ({n_skipped} skipped).")
    if samples:
        total_tokens     = sum(len(s["tokens"])          for s in samples)
        total_boundaries = sum(sum(s["boundary_labels"]) for s in samples)
        pos_rate         = total_boundaries / max(total_tokens, 1)
        ratio            = (1 - pos_rate) / max(pos_rate, 1e-9)
        print(f"  Total words:     {total_tokens:,}")
        print(f"  Boundary rate:   {100*pos_rate:.1f}%  (ratio ≈ {ratio:.1f}:1)")
        print(f"  → Suggested BOUNDARY_LOSS_WEIGHT: ~{round(ratio)}.0")
    return samples, {}


# ── CORPUS_REGISTRY (training data only) ─────────────────────────────────────
# Maps name → sample list. Only loaded corpora appear here.
# Valid keys: "libri", "sbc", "ps"
# BU is evaluation-only and never enters the registry — see below.
CORPUS_REGISTRY = {}

# ── LibriTTS ──────────────────────────────────────────────────────────────────
if LOAD_LIBRI:
    _samples_100, _ = load_label_files_batched(BATCHED_ROOT_LIBRI_100, "libri-100")
    _samples_360, _ = load_label_files_batched(BATCHED_ROOT_LIBRI_360, "libri-360")
    samples_libri = _samples_100 + _samples_360
    print(f"  LibriTTS total: {len(samples_libri):,} samples")
    CORPUS_REGISTRY["libri"] = samples_libri
else:
    print("  ⚠  LOAD_LIBRI=False — 'libri' not in registry.")

# ── SBCSAE ────────────────────────────────────────────────────────────────────
if LOAD_SBC:
    _all_sbc, _ = load_label_files_batched(BATCHED_ROOT_SBC, "sbcsae")
    # Split at load time: SBC001–005 → fixed eval holdout (never in registry)
    samples_sbc_test  = [s for s in _all_sbc
                         if s["sample_id"][:6] in SBC_TEST_IDS]
    samples_sbc_train = [s for s in _all_sbc
                         if s["sample_id"][:6] not in SBC_TEST_IDS]
    print(f"  SBC train: {len(samples_sbc_train):,}  |  SBC eval holdout: {len(samples_sbc_test):,}")
    CORPUS_REGISTRY["sbc"] = samples_sbc_train
else:
    print("  ⚠  LOAD_SBC=False — 'sbc' not in registry.")
    samples_sbc_test = []
    print("  ⚠  samples_sbc_test is empty — test set will be a random split.")

# ── BU Radio News — evaluation only, never in registry ────────────────────────
# Full BU corpus is used as the gold-standard break index evaluation set.
# BU is never added to CORPUS_REGISTRY or any training pool.
# Models trained without BU data can be published freely.
if LOAD_BU:
    samples_bu, _ = load_label_files_batched(BATCHED_ROOT_BU, "bu")
    print(f"  BU eval holdout: {len(samples_bu):,} samples (break index eval only — never in training)")
else:
    samples_bu = []
    print("  ⚠  LOAD_BU=False — BU break index evaluation will be skipped.")

# ── People's Speech ───────────────────────────────────────────────────────────
if LOAD_PS:
    samples_ps, _ = load_label_files_batched(BATCHED_ROOT_PS, "ps")
    CORPUS_REGISTRY["ps"] = samples_ps
else:
    print("  ⚠  LOAD_PS=False — 'ps' not in registry.")

# ── ADD FUTURE CORPORA HERE ───────────────────────────────────────────────────
# if LOAD_NEWCORPUS:
#     samples_newcorpus, _ = load_label_files_batched(BATCHED_ROOT_NEWCORPUS, "newcorpus")
#     CORPUS_REGISTRY["newcorpus"] = samples_newcorpus
# else:
#     print("  ⚠  LOAD_NEWCORPUS=False — 'newcorpus' not in registry.")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n✓ CORPUS_REGISTRY keys:", list(CORPUS_REGISTRY.keys()))
print(f"  SBC eval holdout (SBC001–005):  {len(samples_sbc_test):,} samples  [boundary + intonation]")
print(f"  BU eval holdout  (full corpus): {len(samples_bu):,} samples  [break index]")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Word count verification — train / val / test separate       ║
# ║  Run after Cell 4 (corpus loading). Requires torch.          ║
# ╚══════════════════════════════════════════════════════════════╝
import json, os, torch

SBC_TEST_IDS = {f"SBC{str(n).zfill(3)}" for n in range(1, 6)}
VAL_TARGET_FRAC = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)  # 0.10/0.90 = 0.1111

def split_corpus_words(root, label, exclude_prefix_ids=None):
    """
    Reads batch JSONs, applies the same word-proportional train/val split
    as run_experiment() (seed=42), and returns (train_words, val_words, n_samples).
    """
    exclude_prefix_ids = exclude_prefix_ids or set()
    batch_files = sorted(f for f in os.listdir(root)
                         if f.startswith("batch_") and f.endswith(".json"))
    samples = []
    for fname in batch_files:
        with open(os.path.join(root, fname)) as f:
            batch = json.load(f)
        for sid, data in batch.items():
            if sid[:6] in exclude_prefix_ids:
                continue
            samples.append(len(data["b"]["tokens"]))  # word count for this sample

    corpus_total = sum(samples)
    target_val   = corpus_total * VAL_TARGET_FRAC

    rng  = torch.Generator().manual_seed(SPLIT_SEED)
    idx  = torch.randperm(len(samples), generator=rng).tolist()

    val_words, val_idx_set = 0, set()
    for i in idx:
        if val_words >= target_val:
            break
        val_idx_set.add(i)
        val_words += samples[i]

    train_words = sum(w for j, w in enumerate(samples) if j not in val_idx_set)

    print(f"  {label}")
    print(f"    train:  {train_words:>10,}")
    print(f"    val:    {val_words:>10,}")
    print(f"    pool:   {corpus_total:>10,}  ({len(samples):,} samples)")
    return train_words, val_words

def count_test_words(root, label, keep_prefix_ids=None):
    batch_files = sorted(f for f in os.listdir(root)
                         if f.startswith("batch_") and f.endswith(".json"))
    total, n = 0, 0
    for fname in batch_files:
        with open(os.path.join(root, fname)) as f:
            batch = json.load(f)
        for sid, data in batch.items():
            if keep_prefix_ids is None or any(sid.startswith(p) for p in keep_prefix_ids):
                total += len(data["b"]["tokens"])
                n += 1
    print(f"  {label}")
    print(f"    test:   {total:>10,}  ({n} samples)")
    return total

print("=== WORD COUNT VERIFICATION ===\n")

l100_tr, l100_vl = split_corpus_words(BATCHED_ROOT_LIBRI_100, "LibriTTS-100")
l360_tr, l360_vl = split_corpus_words(BATCHED_ROOT_LIBRI_360, "LibriTTS-360")
ps_tr,   ps_vl   = split_corpus_words(BATCHED_ROOT_PS,        "People's Speech")
sbc_tr,  sbc_vl  = split_corpus_words(BATCHED_ROOT_SBC,       "SBCSAE (SBC006–060)",
                                       exclude_prefix_ids=SBC_TEST_IDS)
sbc_te           = count_test_words(BATCHED_ROOT_SBC, "SBCSAE (SBC001–005 test)",
                                    keep_prefix_ids=SBC_TEST_IDS)
bu_te = count_test_words(BATCHED_ROOT_BU, "BU Radio News (test)", keep_prefix_ids=None)

print(f"\n{'─'*45}")
print(f"  {'':25s} {'Train':>10} {'Val':>10} {'Test':>10}")
print(f"  {'LibriTTS-100':25s} {l100_tr:>10,} {l100_vl:>10,} {'—':>10}")
print(f"  {'LibriTTS-360':25s} {l360_tr:>10,} {l360_vl:>10,} {'—':>10}")
print(f"  {'People\'s Speech':25s} {ps_tr:>10,} {ps_vl:>10,} {'—':>10}")
print(f"  {'SBCSAE (SBC006–060)':25s} {sbc_tr:>10,} {sbc_vl:>10,} {'—':>10}")
print(f"  {'SBCSAE (SBC001–005)':25s} {'—':>10} {'—':>10} {sbc_te:>10,}")
print(f"  {'BU Radio News':25s} {'—':>10} {'—':>10} {bu_te:>10,}")
print(f"  {'─'*45}")
print(f"  {'TOTAL':25s} {l100_tr+l360_tr+ps_tr+sbc_tr:>10,} {l100_vl+l360_vl+ps_vl+sbc_vl:>10,} {sbc_te+bu_te:>10,}")
print(f"\nHparams says: train=7,035,653  val=879,670")

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Data split verification — full table                        ║
# ║                                                              ║
# ║  Train/val figures: read from batch JSONs on Drive,          ║
# ║  replicating the word-proportional split in run_experiment() ║
# ║                                                              ║
# ║  SBCSAE test (21,304): verified independently from raw .trn  ║
# ║  transcript files                                            ║
# ║                                                              ║
# ║  BU test (30,796): read from batch JSONs on Drive            ║
# ╚══════════════════════════════════════════════════════════════╝
import json, os, torch

SBC_TEST_IDS     = {f"SBC{str(n).zfill(3)}" for n in range(1, 6)}
VAL_TARGET_FRAC  = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)  # 0.10/0.90 = 0.1111

# ── Helpers ───────────────────────────────────────────────────────────────────

def split_corpus(root, label, exclude_ids=None):
    """
    Read batch JSONs, exclude any sample whose ID prefix is in exclude_ids,
    then apply the same word-proportional train/val split as run_experiment().
    Returns (train_words, val_words).
    """
    exclude_ids = exclude_ids or set()
    batch_files = sorted(f for f in os.listdir(root)
                         if f.startswith("batch_") and f.endswith(".json"))
    samples = []
    for fname in batch_files:
        with open(os.path.join(root, fname)) as f:
            batch = json.load(f)
        for sid, data in batch.items():
            if sid[:6] in exclude_ids:
                continue
            samples.append(len(data["b"]["tokens"]))

    corpus_total = sum(samples)
    target_val   = corpus_total * VAL_TARGET_FRAC

    rng = torch.Generator().manual_seed(SPLIT_SEED)
    idx = torch.randperm(len(samples), generator=rng).tolist()

    val_words, val_idx_set = 0, set()
    for i in idx:
        if val_words >= target_val:
            break
        val_idx_set.add(i)
        val_words += samples[i]

    train_words = sum(w for j, w in enumerate(samples) if j not in val_idx_set)
    return train_words, val_words


def count_all(root, label):
    """Count all tokens in a batch root (for test-only corpora)."""
    batch_files = sorted(f for f in os.listdir(root)
                         if f.startswith("batch_") and f.endswith(".json"))
    total = 0
    for fname in batch_files:
        with open(os.path.join(root, fname)) as f:
            batch = json.load(f)
        for sid, data in batch.items():
            total += len(data["b"]["tokens"])
    return total


# ── Compute all figures ───────────────────────────────────────────────────────

l100_tr, l100_vl = split_corpus(BATCHED_ROOT_LIBRI_100, "LibriTTS-100")
l360_tr, l360_vl = split_corpus(BATCHED_ROOT_LIBRI_360, "LibriTTS-360")
ps_tr,   ps_vl   = split_corpus(BATCHED_ROOT_PS,        "People's Speech")
sbc_tr,  sbc_vl  = split_corpus(BATCHED_ROOT_SBC,       "SBCSAE (SBC006-060)",
                                 exclude_ids=SBC_TEST_IDS)

# SBCSAE test: 21,304 unique words, verified from raw .trn transcript files.
# The batch JSON approach gives 42,330 here due to overlapping IU windows —
# that figure is not used.
sbc_te = 21_304

bu_te = count_all(BATCHED_ROOT_BU, "BU Radio News")

# ── Print table ───────────────────────────────────────────────────────────────

W = 12
D = "—"
SEP = "─" * 57

def row(label, tr, vl, te):
    tr_s = f"{tr:>{W},}" if isinstance(tr, int) else f"{D:>{W}}"
    vl_s = f"{vl:>{W},}" if isinstance(vl, int) else f"{D:>{W}}"
    te_s = f"{te:>{W},}" if isinstance(te, int) else f"{D:>{W}}"
    print(f"  {label:<26}{tr_s}{vl_s}{te_s}")

print(f"\n  {'':26}{'Train':>{W}}{'Val':>{W}}{'Test':>{W}}")
print(f"  {SEP}")
row("LibriTTS-100",        l100_tr, l100_vl, None)
row("LibriTTS-360",        l360_tr, l360_vl, None)
row("People's Speech",     ps_tr,   ps_vl,   None)
row("SBCSAE (SBC006–060)", sbc_tr,  sbc_vl,  None)
row("SBCSAE (SBC001–005)", None,    None,    sbc_te)
row("BU Radio News",       None,    None,    bu_te)
print(f"  {SEP}")
row("TOTAL",
    l100_tr + l360_tr + ps_tr + sbc_tr,
    l100_vl + l360_vl + ps_vl + sbc_vl,
    sbc_te  + bu_te)

print(f"\n  Hparams reference: train=7,035,653  val=879,670")
print(f"  (Δ of ±7 words is rounding in the word-proportional split — expected)")